# CodeX Project - Feature Engineering


This notebook creates required engineered features from `dataset/survey_results_cleaned.csv` as per `feature_engineering_instructions.pdf`.


In [16]:
import pandas as pd

In [17]:
input_path = 'dataset/survey_results_cleaned.csv'
output_path = 'dataset/survey_results_feature_engineered.csv'

df = pd.read_csv(input_path)
print('Input shape:', df.shape)
df.head()


Input shape: (29991, 17)


,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range
0,R00001,30,M,Urban,Working Professional,<10L,3-4 times,Newcomer,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150
1,R00002,46,F,Metro,Working Professional,> 35L,5-7 times,Established,Medium (500 ml),2 to 4,Quality,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),Social (eg. Parties),200-250
2,R00003,41,F,Rural,Working Professional,> 35L,3-4 times,Newcomer,Medium (500 ml),2 to 4,Availability,Traditional,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",200-250
3,R00004,33,F,Urban,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),0 to 1,Brand Reputation,Exotic,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",150-200
4,R00005,23,M,Metro,Student,Not Reported,3-4 times,Established,Medium (500 ml),0 to 1,Availability,Traditional,Online,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100


## Step 1: Create `age_group` and remove `age`


In [18]:
bins = [17, 25, 35, 45, 55, 70, float('inf')]
labels = ['18-25', '26-35', '36-45', '46-55', '56-70', '70+']

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

print('Age group counts before removing age:')
print(df['age_group'].value_counts(dropna=False).sort_index())


Age group counts before removing age:
age_group
18-25    10468
26-35     9093
36-45     5972
46-55     2966
56-70     1492
70+          0
Name: count, dtype: int64


## Step 2: Create `cf_ab_score`


In [19]:
freq_map = {
    '0-2 times': 1,
    '3-4 times': 2,
    '5-7 times': 3
}
awareness_map = {
    '0 to 1': 1,
    '2 to 4': 2,
    'above 4': 3
}

frequency_score = df['consume_frequency(weekly)'].map(freq_map)
awareness_score = df['awareness_of_other_brands'].map(awareness_map)

df['cf_ab_score'] = (frequency_score / (awareness_score + frequency_score)).round(2)

print(df['cf_ab_score'].describe())


count    29991.000000
mean         0.537350
std          0.141866
min          0.250000
25%          0.500000
50%          0.500000
75%          0.670000
max          0.750000
Name: cf_ab_score, dtype: float64


## Step 3: Create `zas_score`


In [20]:
zone_map = {
    'Urban': 3,
    'Metro': 4,
    'Rural': 1,
    'Semi-Urban': 2
}
income_map = {
    '<10L': 1,
    '10L - 15L': 2,
    '16L - 25L': 3,
    '26L - 35L': 4,
    '> 35L': 5,
    'Not Reported': 0
}

zone_score = df['zone'].map(zone_map)
income_score = df['income_levels'].map(income_map)

df['zas_score'] = (zone_score * income_score).astype('Int64')

print(df['zas_score'].describe())
print('-------------------------')
print(df['zas_score'].nunique())

count     29991.0
mean     6.096529
std      5.517959
min           0.0
25%           0.0
50%           6.0
75%           9.0
max          20.0
Name: zas_score, dtype: Float64
-------------------------
14


## Step 4: Create `bsi` (Brand Switching Indicator)


In [21]:
df['bsi'] = ((df['current_brand'] != 'Established') &
             (df['reasons_for_choosing_brands'].isin(['Price', 'Quality']))).astype(int)

print(df['bsi'].value_counts())


bsi
0    20816
1     9175
Name: count, dtype: int64


## Final Cleaning: Remove Logical Outliers


In [22]:
# Logical inconsistency from instructions:
# remove records where occupation is Student in age_group 56-70 (and 70+ if present)
outlier_mask = (df['occupation'] == 'Student') & (df['age_group'].isin(['56-70', '70+']))
print('Logical outliers to remove:', int(outlier_mask.sum()))

df = df.loc[~outlier_mask].copy()

Logical outliers to remove: 35


## Finalize and Export


In [23]:
# Remove original age column as instructed
if 'age' in df.columns:
    df = df.drop(columns=['age'])

# Convert age_group to string for clean CSV export
if str(df['age_group'].dtype) == 'category':
    df['age_group'] = df['age_group'].astype(str)

validation = {
    'rows_after_engineering': int(df.shape[0]),
    'columns_after_engineering': int(df.shape[1]),
    'age_removed': ('age' not in df.columns),
    'age_group_nulls': int(df['age_group'].isna().sum()),
    'cf_ab_score_nulls': int(df['cf_ab_score'].isna().sum()),
    'zas_score_nulls': int(df['zas_score'].isna().sum()),
    'bsi_nulls': int(df['bsi'].isna().sum()),
    'student_56_70_remaining': int(((df['occupation']=='Student') & (df['age_group'].isin(['56-70','70+']))).sum())
}
validation


{'rows_after_engineering': 29956,
 'columns_after_engineering': 20,
 'age_removed': True,
 'age_group_nulls': 0,
 'cf_ab_score_nulls': 0,
 'zas_score_nulls': 0,
 'bsi_nulls': 0,
 'student_56_70_remaining': 0}

In [24]:
df.to_csv(output_path, index=False)
print('Saved:', output_path)


Saved: dataset/survey_results_feature_engineered.csv
